# Session 2: Cypher — Da Basi a Query Reali

Benvenuti alla Sessione 2! Qui approfondiremo Cypher sul dataset Org Graph caricato in Sessione 1.

**Cosa faremo:**
1. Connessione a Neo4j e ricarica dataset
2. Pattern MATCH avanzati (path variables, multi-hop)
3. MERGE vs CREATE: idempotenza
4. Filtri WHERE (AND, OR, IN, CONTAINS)
5. Aggregazioni e subquery
6. Date, liste, map
7. Query tuning con PROFILE / EXPLAIN
8. APOC essentials
9. Esercizi

**Dataset:** Org Graph (238 persone, 93 progetti, 55 skill)


## 1. Setup e Connessione


Installiamo le dipendenze e connettiamoci al database Neo4j.
Il driver v6+ supporta `execute_query()` che useremo in tutto il notebook.


In [ ]:
# Installa le dipendenze necessarie
%pip install neo4j python-dotenv pandas -q


In [ ]:
# Import librerie
from neo4j import GraphDatabase
import pandas as pd
import os
from dotenv import load_dotenv

# Carica credenziali dal file .env (root del progetto)
load_dotenv("../../.env")

# Legge URI, username, password
URI  = os.getenv("NEO4J_URI")
USER = os.getenv("NEO4J_USERNAME")
PASS = os.getenv("NEO4J_PASSWORD")

# Crea il driver (pool di connessioni thread-safe)
driver = GraphDatabase.driver(URI, auth=(USER, PASS))

# Verifica la connessione
driver.verify_connectivity()
print("Connesso a Neo4j!")


## 2. Verifica Dati e Ricarica


Controlliamo se il dataset Org Graph è già presente nel database.
Se sì (da Sessione 1), saltiamo la ricarica. Altrimenti eseguiamo il caricamento rapido qui sotto.


In [ ]:
# Verifica se il database contiene dati
query = 'MATCH (p:Person) RETURN count(p) AS cnt'
records, _, _ = driver.execute_query(query)
persone = records[0]["cnt"]
print(f"Persone trovate nel database: {persone}")
if persone == 0:
    print("Nessun dato. Eseguire le celle di caricamento qui sotto.")
else:
    print(f"Dataset già presente ({persone} persone). Saltare la ricarica.")


In [ ]:
# Solo se necessario: carica i dati dal CSV nel database
# Usa UNWIND batching (7 query in 1 secondo) + MERGE (idempotente)
import csv, time
start = time.time()
with open("../../data/org_graph.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
people = []; sups = {}; projs = {}; skills = set()
rel_skill = []; rel_work = []; rel_report = []
core = {"employee_id","name","gender","discipline","department",
        "code","location","title","supervisor_id","supervisor_name",
        "project_id","project_name","role"}
skill_cols = [k for k in rows[0].keys() if k not in core]
for row in rows:
    eid = int(row["employee_id"])
    people.append({"id":eid,"name":row["name"],
        "gender":row["gender"],"disc":row["discipline"],"dept":row["department"],
        "code":row["code"],"loc":row["location"],"title":row["title"]})
    if row["supervisor_id"]:
        sid = int(row["supervisor_id"])
        sups[sid] = row["supervisor_name"]
        rel_report.append({"eid":eid,"sid":sid})
    if row["project_id"]:
        pid = int(row["project_id"])
        projs[pid] = row["project_name"]
        rel_work.append({"eid":eid,"pid":pid,"role":row["role"]})
    for s in skill_cols:
        if row[s].strip():
            skills.add(s)
            rel_skill.append({"eid":eid,"skill":s,"prof":row[s].strip()})
print(f"Collected: {len(people)} people, {len(sups)} supervisors, {len(projs)} projects, {len(skills)} skills")
with driver.session() as session:
    session.run("UNWIND $b AS p MERGE (x:Person {employee_id:p.id}) SET x.name=p.name,x.gender=p.gender,x.discipline=p.disc,x.department=p.dept,x.code=p.code,x.location=p.loc,x.title=p.title", b=people)
    print(" [1/7] People:", len(people))
    sl = [{"sid":s,"name":n} for s,n in sups.items()]
    session.run("UNWIND $b AS s MERGE (x:Supervisor {supervisor_id:s.sid}) SET x.name=s.name", b=sl)
    print(" [2/7] Supervisors:", len(sl))
    pl = [{"pid":p,"name":n} for p,n in projs.items()]
    session.run("UNWIND $b AS p MERGE (x:Project {project_id:p.pid}) SET x.name=p.name", b=pl)
    print(" [3/7] Projects:", len(pl))
    sbl = [{"name":s} for s in sorted(skills)]
    session.run("UNWIND $b AS s MERGE (x:Skill {name:s.name})", b=sbl)
    print(" [4/7] Skills:", len(sbl))
    session.run("UNWIND $b AS r MATCH (p:Person {employee_id:r.eid}), (s:Skill {name:r.skill}) MERGE (p)-[h:HAS_SKILL]->(s) SET h.proficiency=r.prof", b=rel_skill)
    print(" [5/7] HAS_SKILL:", len(rel_skill))
    session.run("UNWIND $b AS r MATCH (p:Person {employee_id:r.eid}), (x:Project {project_id:r.pid}) MERGE (p)-[w:WORKED_ON]->(x) SET w.role=r.role", b=rel_work)
    print(" [6/7] WORKED_ON:", len(rel_work))
    session.run("UNWIND $b AS r MATCH (p:Person {employee_id:r.eid}), (s:Supervisor {supervisor_id:r.sid}) MERGE (p)-[:REPORTS_TO]->(s)", b=rel_report)
    print(" [7/7] REPORTS_TO:", len(rel_report))
print(f"Graph caricato in {time.time()-start:.1f}s")


## 3. Esplorare il Grafo


Prima di scrivere query complesse, esploriamo la struttura del grafo.
Vediamo quali nodi esistono, quante realtà ci sono e come appaiono i dati.


In [ ]:
# Esploriamo il modello: quanti nodi per ogni label?
for label in ["Person", "Supervisor", "Skill", "Project"]:
    records, _, _ = driver.execute_query(f"MATCH (:{label}) RETURN count(*) AS cnt")
    cnt = records[0]["cnt"]
    print(f"{label:12s} | {cnt} nodi")


In [ ]:
# Esempio di nodo Person per vedere le proprietà disponibili
query = "MATCH (p:Person) RETURN p.name, p.location, p.title, p.department LIMIT 5"
records, _, _ = driver.execute_query(query)
for r in records:
    print(f"{r["p.name"]:20s} | {r["p.location"]:15s} | {r["p.title"]:25s} | {r["p.department"]}")


In [ ]:
# Esempio di relazione: chi riporta a quale supervisor
query = """
MATCH (p:Person)-[:REPORTS_TO]->(s:Supervisor)
RETURN p.name AS impiegato, s.name AS supervisor
LIMIT 10
"""
records, _, _ = driver.execute_query(query)
for r in records:
    print(f"{r["impiegato"]:20s} -> {r["supervisor"]}")


## 4. Pattern MATCH Avanzati


**Path a lunghezza variabile** con `[*min..max]` permette di navigare più hop:
- `-[:WORKED_ON*]-(b)` : profondità illimitata attraverso progetti

**shortestPath()** : cammino minimo tra due nodi.

**Path variables** : l'intero percorso assegnato a una variabile per analizzarlo.


In [ ]:
# --- ShortestPath: cammino minimo tra due impiegati ---
# Prima troviamo due persone VALIDE che hanno progetti in comune
# Altrimenti shortestPath non trova alcun percorso
query = """
MATCH (p:Person)-[:WORKED_ON]->(:Project)<-[:WORKED_ON]-(q:Person)
WHERE id(p) < id(q)
WITH p, q, count(*) AS progetti_comuni
WHERE progetti_comuni >= 1
RETURN p.employee_id AS id_a, p.name AS nome_a,
       q.employee_id AS id_b, q.name AS nome_b,
       progetti_comuni
ORDER BY progetti_comuni DESC
LIMIT 5
"""
records, _, _ = driver.execute_query(query)
for r in records:
    print(f"{r["nome_a"]:20s} (id:{r["id_a"]}) <-> {r["nome_b"]:20s} (id:{r["id_b"]}) = {r["progetti_comuni"]} progetti comuni")


In [ ]:
# --- ShortestPath tra le prime due persone trovate ---
# Usiamo i primi due ID validi dalla query precedente
query = """
MATCH (a:Person {employee_id: 1}), (b:Person {employee_id: 100})
MATCH path = shortestPath((a)-[:WORKED_ON*]-(b))
RETURN [n IN nodes(path) | n.name] AS percorso,
       length(path) AS hop
"""
records, _, _ = driver.execute_query(query)
if records:
    for r in records:
        print("Percorso (" + str(r["hop"]) + " hop):", " -> ".join(r["percorso"]))
else:
    # Se employee_id 1 e 100 non esistono o non hanno percorso, usiamo gli ID trovati sopra
    print("Nessun percorso tra employee_id 1 e 100. Verifica gli ID validi dalla cella precedente.")


In [ ]:
# --- Path attraverso supervisor (Person -> Supervisor) ---
# REPORTS_TO collega Person a Supervisor (non Person a Person)
# Per vedere tutti in una gerarchia, navighiamo con 1 salto
query = """
MATCH (p:Person)-[:REPORTS_TO]->(s:Supervisor)
RETURN p.name AS impiegato, s.name AS supervisor
ORDER BY s.name, p.name
LIMIT 20
"""
records, _, _ = driver.execute_query(query)
for r in records:
    print(f"{r["impiegato"]:20s} -> {r["supervisor"]}")


## 4. MERGE vs CREATE: Idempotenza


- **CREATE**: inserisce sempre. Se il nodo esiste (UNIQUE constraint), fallisce.
- **MERGE**: cerca il pattern. Se non esiste, lo crea.
- Idempotente: eseguire N volte produce lo stesso risultato.
- `ON MATCH SET` / `ON CREATE SET` per comportamenti distinti.


In [ ]:
# --- MERGE idempotente ---
# La prima volta crea, la seconda volta trova e basta
query = """
MERGE (p:Person {employee_id: 999})
  SET p.name = "Test User", p.location = "Test"
RETURN p.employee_id, p.name, p.location
"""
records, _, _ = driver.execute_query(query)
print("1a esecuzione:", records[0]["name"])
records, _, _ = driver.execute_query(query)
print("2a esecuzione (idempotente):", records[0]["name"])

# Pulizia
driver.execute_query('MATCH (p:Person {employee_id: 999}) DETACH DELETE p')
print("Pulito")


In [ ]:
# --- MERGE su relazione con ON CREATE/MATCH SET ---
query = """
MATCH (p:Person {employee_id: 1}), (s:Skill {name: 'Python'})
MERGE (p)-[h:HAS_SKILL]->(s)
  ON CREATE SET h.proficiency = 'Senior'
  ON MATCH SET h.proficiency = 'Senior'
RETURN p.name, s.name, h.proficiency
"""
records, _, _ = driver.execute_query(query)
print(records[0]["p.name"], "-", records[0]["s.name"], "-", records[0]["h.proficiency"])


## 5. Filtri WHERE Avanzati


AND, OR, NOT, IN [...], CONTAINS, STARTS WITH, ENDS WITH, IS NULL.
Possono essere usati sia nel WHERE che direttamente nel MATCH.


In [ ]:
# --- AND/OR con IN e STARTS WITH ---
query = """
MATCH (p:Person)
WHERE (p.location IN ["Roma", "Milano"])
  AND p.name STARTS WITH "M"
RETURN p.name, p.location, p.title
ORDER BY p.name
LIMIT 10
"""
records, _, _ = driver.execute_query(query)
for r in records:
    print(r["p.name"], "|", r["p.location"], "|", r["p.title"])


In [ ]:
# --- CONTAINS: ricerca testuale su skill ---
query = """
MATCH (s:Skill)
WHERE s.name CONTAINS "cloud"
RETURN s.name AS skill
ORDER BY skill
"""
records, _, _ = driver.execute_query(query)
print("Skill con \"cloud\":", len(records))
for r in records:
    print(" -", r["skill"])


## 6. Aggregazioni


count(*), collect(), avg(), sum(), min(), max().
Le colonne non aggregate sono automaticamente grouping key.


In [ ]:
# --- Skill più diffuse ---
query = """
MATCH (p:Person)-[:HAS_SKILL]->(s:Skill)
RETURN s.name AS skill, count(*) AS persone
ORDER BY persone DESC
LIMIT 10
"""
records, _, _ = driver.execute_query(query)
for r in records:
    print(r["skill"], "|", r["persone"])


In [ ]:
# --- Proficiency media per skill ---
# h.proficiency è la proprietà sulla relazione HAS_SKILL
query = """
MATCH (p:Person)-[h:HAS_SKILL]->(s:Skill)
RETURN s.name AS skill,
       round(avg(h.proficiency), 1) AS media,
       count(*) AS persone
ORDER BY media DESC
LIMIT 10
"""
records, _, _ = driver.execute_query(query)
for r in records:
    print(r["skill"], "| media:", r["media"], "|", r["persone"], "persone")


In [ ]:
# --- COLLECT: lista progetti per persona ---
query = """
MATCH (p:Person)-[:WORKED_ON]->(proj:Project)
RETURN p.name AS persona,
       collect(proj.name) AS progetti,
       count(proj) AS num
ORDER BY num DESC
LIMIT 10
"""
records, _, _ = driver.execute_query(query)
for r in records:
    proj_list = ", ".join(r["progetti"])
    print(r["persona"], "|", r["num"], "progetti:", proj_list)


## 7. Ordinamento e Paginazione


ORDER BY ASC/DESC (anche multipli), LIMIT n, SKIP n per paginazione.


In [ ]:
# --- Paginazione: seconda pagina (record 11-20) ---
query = """
MATCH (p:Person)
RETURN p.name AS nome, p.location AS location
ORDER BY nome
SKIP 10 LIMIT 10
"""
records, _, _ = driver.execute_query(query)
for r in records:
    print(r["nome"], "|", r["location"])


## 8. Subquery con CALL { ... }


CALL { subquery } permette di annidare query. UNION combina risultati.
Le subquery correlate possono riferire variabili esterne.


In [ ]:
# --- UNION: skill e progetti in una lista ---
query = """
CALL { MATCH (s:Skill) RETURN s.name AS nome, 'Skill' AS tipo }
UNION
CALL { MATCH (p:Project) RETURN p.name AS nome, 'Progetto' AS tipo }
RETURN nome, tipo
ORDER BY nome
LIMIT 20
"""
records, _, _ = driver.execute_query(query)
for r in records:
    print(r["nome"], "|", r["tipo"])


In [ ]:
# --- Subquery correlate: conteggi multipli per persona ---
# Ogni CALL { } si riferisce alla persona corrente (p)
query = """
MATCH (p:Person)
RETURN p.name AS nome,
       CALL { MATCH (p)-[:HAS_SKILL]->(s:Skill) RETURN count(*) } AS num_skill,
       CALL { MATCH (p)-[:WORKED_ON]->(proj:Project) RETURN count(*) } AS num_progetti
ORDER BY num_skill DESC
LIMIT 10
"""
records, _, _ = driver.execute_query(query)
for r in records:
    print(r["nome"], "| skill:", r["num_skill"], "| progetti:", r["num_progetti"])


## 9. Date, Liste e Mappe


In [ ]:
# date() = data odierna, date('2026-07-15') = data specifica
query = "RETURN date() AS oggi, date('2026-07-15') AS prima_sessione"
records, _, _ = driver.execute_query(query)
print("Oggi:", records[0]["oggi"])
print("Prima sessione:", records[0]["prima_sessione"])


In [ ]:
# range(start, end) + UNWIND = genera righe da una lista
query = """
UNWIND range(2020, 2026) AS anno
RETURN anno
"""
records, _, _ = driver.execute_query(query)
for r in records:
    print(r['anno'])


In [ ]:
# List comprehension: [x IN list WHERE cond | expr]
# Filtra numeri > 2 e moltiplica per 10
query = """
WITH [1, 2, 3, 4, 5] AS numeri
RETURN [x IN numeri WHERE x > 2 | x * 10] AS trasformati
"""
records, _, _ = driver.execute_query(query)
print("Trasformati:", records[0]["trasformati"])


## 10. Query Tuning: PROFILE ed EXPLAIN


EXPLAIN = piano stimato (non esegue).
PROFILE = esegue e mostra dbHits, rows.
NodeIndexSeek = buono (usa indice). AllNodesScan = cattivo (scan totale).


In [ ]:
# EXPLAIN: piano stimato senza esecuzione
query = "EXPLAIN MATCH (p:Person)-[:HAS_SKILL]->(s:Skill) WHERE p.location = 'Roma' RETURN p.name, s.name"
records, summary, _ = driver.execute_query(query)
print("=== PIANO STIMATO ===")
for op in summary.plan.operators:
    print(' ', op)


In [ ]:
# PROFILE: esecuzione con metriche reali
query = "PROFILE MATCH (p:Person {location: 'Roma'})-[:HAS_SKILL]->(s:Skill) RETURN p.name, s.name"
records, summary, _ = driver.execute_query(query)
print("=== RISULTATI ===")
for r in records:
    print(r["p.name"], "|", r["s.name"])


## 11. APOC: Procedure Standard


APOC = libreria ufficiale di procedure. Su AuraDB molte sono già disponibili.
apoc.coll.*, apoc.text.*, apoc.load.csv, apoc.periodic.iterate.


In [ ]:
# Verifica disponibilita APOC
try:
    records, _, _ = driver.execute_query('RETURN apoc.version() AS versione')
    print("APOC versione:", records[0]["versione"])
except Exception as e:
    print("APOC non disponibile:", e)


In [ ]:
# apoc.coll.zip = unisce due liste come Python zip()
query = "RETURN apoc.coll.zip([1, 2, 3], ['a', 'b', 'c']) AS zippato"
records, _, _ = driver.execute_query(query)
print("Zippato:", records[0]["zippato"])


In [ ]:
# apoc.text.clean = normalizza testo (rimuove accenti, speciali)
query = "RETURN apoc.text.clean('Cypher Query Language!') AS pulito"
records, _, _ = driver.execute_query(query)
print("Pulito:", records[0]["pulito"])


## 12. Esercizi


Scrivete query Cypher per questi problemi sul dataset Org Graph.
Potete usare driver.execute_query() qui o Neo4j Browser.

**Proprietà del modello:**
- Person: employee_id, name, location, title, department
- Skill: name
- Project: project_id, name
- HAS_SKILL: proficiency
- WORKED_ON: role


### Esercizio 1 — Skill più diffuse


Trovate le 5 skill più diffuse tra i dipendenti.


In [ ]:
query = """
MATCH (p:Person)-[:HAS_SKILL]->(s:Skill)
RETURN s.name AS skill, count(*) AS persone
ORDER BY persone DESC
LIMIT 5
"""
records, _, _ = driver.execute_query(query)
for r in records:
    print(r["skill"], "-", r["persone"], "persone")


### Esercizio 2 — Progetti per persona


Per ogni persona, contate i progetti. Mostrate le prime 10.


In [ ]:
query = """
MATCH (p:Person)-[:WORKED_ON]->(proj:Project)
RETURN p.name AS persona, count(proj) AS progetti
ORDER BY progetti DESC
LIMIT 10
"""
records, _, _ = driver.execute_query(query)
for r in records:
    print(r["persona"], "-", r["progetti"], "progetti")


### Esercizio 3 — Cammino minimo


shortestPath tra due impiegati a scelta via progetti condivisi.


In [ ]:
query = """
MATCH (a:Person {employee_id: 1}), (b:Person {employee_id: 50})
MATCH path = shortestPath((a)-[:WORKED_ON*]-(b))
RETURN [n IN nodes(path) | n.name] AS percorso,
       length(path) AS hop
"""
records, _, _ = driver.execute_query(query)
for r in records:
    print(" -> ".join(r["percorso"]), "(", r["hop"], "hop)")


### Esercizio 4 — Skill overlap


Coppie di impiegati con almeno 3 skill in comune.


In [ ]:
query = """
MATCH (p1:Person)-[:HAS_SKILL]->(s:Skill)<-[:HAS_SKILL]-(p2:Person)
WHERE id(p1) < id(p2)
WITH p1.name AS p1n, p2.name AS p2n, collect(s.name) AS comuni
WHERE size(comuni) >= 3
RETURN p1n, p2n, comuni
ORDER BY p1n
LIMIT 20
"""
records, _, _ = driver.execute_query(query)
for r in records:
    print(r["p1n"], "|", r["p2n"], "|", ", ".join(r["comuni"]))


### Esercizio 5 — Subquery correlate


Nome, skill e progetti per ogni persona via CALL { }.


In [ ]:
query = """
MATCH (p:Person)
RETURN p.name AS nome,
       CALL { MATCH (p)-[:HAS_SKILL]->(s:Skill) RETURN count(*) } AS num_skill,
       CALL { MATCH (p)-[:WORKED_ON]->(proj:Project) RETURN count(*) } AS num_progetti
ORDER BY nome
LIMIT 15
"""
records, _, _ = driver.execute_query(query)
for r in records:
    print(r["nome"], "| skill:", r["num_skill"], "| progetti:", r["num_progetti"])


### Esercizio 6 — PROFILE


Analizzate una query con PROFILE. Cercate operatori inefficienti.


In [ ]:
query = """PROFILE
MATCH (p1:Person)-[:HAS_SKILL]->(s:Skill)<-[:HAS_SKILL]-(p2:Person)
WHERE id(p1) < id(p2)
RETURN p1.name, p2.name, count(s) AS skill_comuni
ORDER BY skill_comuni DESC
LIMIT 10
"""
records, summary, _ = driver.execute_query(query)
print("=== OPERATORI ===")
for op in summary.plan.operators:
    print(' ', op)
print("\n=== RISULTATI ===")
for r in records:
    print(r["p1.name"], "|", r["p2.name"], "|", r["skill_comuni"], "skill")


## Riepilogo


| Concetto | Esempio |
|----------|--------|
| Path variabile | (a)-[:WORKED_ON*]-(b) |
| ShortestPath | shortestPath((a)-[:WORKED_ON*]-(b)) |
| MERGE idempotente | MERGE (p:Person {id: n}) SET ... |
| Filtri WHERE | location IN [...] AND name CONTAINS ... |
| Aggregazioni | count(*), collect(), avg() |
| Subquery | CALL { MATCH ... } |
| PROFILE/EXPLAIN | diagnostica performance |
| APOC | apoc.coll.*, apoc.text.* |


In [ ]:
driver.close()
print("Connessione chiusa")
